# Cada bloco vai para qual fábrica?

**Fonte:** base sintética da palestra (dados ilustrativos, sem relação com empresa real).

**Neste notebook:** o mesmo problema da planilha `03_blocos_fabricas.xlsx`, agora com variáveis **binárias** e **índices**. Parte 1: os 30 blocos da planilha (o resultado tem de bater com o Excel). Parte 2: os 120 blocos da base, que não cabem no Solver do Excel. Parte 3: cenários de cota e de uma nova fábrica.

**Sua tarefa:** completar as restrições do modelo. Onde aparecer `____`, escreva o número certo (por exemplo, `10`) ou o nome do dado (por exemplo, `mao_obra_pinus`); os dois funcionam. A primeira restrição vem pronta, como exemplo.

**Se aparecer `NameError: name '____' is not defined`**, ainda falta preencher alguma lacuna.

**Como rodar:** menu **Ambiente de execução → Executar tudo**. Os passos são os mesmos da planilha:
**estimar → modelar → resolver → comparar**.

## 0. Preparar o ambiente

Instala o Pyomo (a linguagem do modelo) e o HiGHS (o *solver*). O Colab apaga tudo ao fechar a aba,
então esta célula roda sempre que o notebook é aberto.

In [ ]:
%pip install -q pyomo highspy

Abaixo ficam três ferramentas para resolver e ler o resultado em tabelas.
**Você não precisa entender este código:** basta rodar.

In [ ]:
import pyomo.environ as pyo
import pandas as pd


def br(numero, casas=2):
    """Escreve um número no formato brasileiro: 1.234,56."""
    texto = f"{numero:,.{casas}f}"
    return texto.replace(",", "X").replace(".", ",").replace("X", ".")


def resolver(modelo):
    """Chama o HiGHS, mostra o status e devolve o valor do objetivo."""
    resultado = pyo.SolverFactory("appsi_highs").solve(modelo)
    print("Status:", resultado.solver.termination_condition)
    objetivo = next(modelo.component_objects(pyo.Objective, active=True))
    print("Objetivo:", br(pyo.value(objetivo)))
    return pyo.value(objetivo)


def ler_variaveis(modelo):
    """Tabela com o valor de cada variável de decisão."""
    linhas = []
    for variavel in modelo.component_objects(pyo.Var, active=True):
        for indice in variavel:
            nome = variavel.name if indice is None else f"{variavel.name}[{indice}]"
            linhas.append({"variável": nome, "valor": round(pyo.value(variavel[indice]), 2)})
    return pd.DataFrame(linhas)


def ler_restricoes(modelo):
    """Tabela com o uso, o limite e a folga de cada restrição (como a aba 4_comparar)."""
    linhas = []
    for restricao in modelo.component_objects(pyo.Constraint, active=True):
        for indice in restricao:
            r = restricao[indice]
            usado = pyo.value(r.body)
            if r.equality:
                sinal, limite, folga = "=", pyo.value(r.upper), usado - pyo.value(r.upper)
            elif r.has_ub():
                sinal, limite, folga = "<=", pyo.value(r.upper), pyo.value(r.upper) - usado
            else:
                sinal, limite, folga = ">=", pyo.value(r.lower), usado - pyo.value(r.lower)
            nome = restricao.name if indice is None else f"{restricao.name}[{indice}]"
            linhas.append({
                "restrição": nome,
                "usado": round(usado, 2),
                "sinal": sinal,
                "limite": round(limite, 2),
                "folga": round(folga, 2),
                "limitou?": "Sim" if abs(folga) < 1e-6 else "Não",
            })
    return pd.DataFrame(linhas)

## Enunciado

Uma empresa tem duas fábricas, A e B, e precisa decidir para onde vai a madeira de cada bloco de plantio.
Cada bloco vai **inteiro** para uma fábrica só. Quanto mais longe, mais caro o frete; a medida é a
**DMT ponderada pela área** (km), em que blocos grandes pesam mais. As fábricas têm tamanhos diferentes:
a A precisa de pelo menos **40%** da área e a B de pelo menos **55%**.

**A pergunta:** para qual fábrica vai cada bloco, para a DMT ser a menor possível e as duas cotas serem atendidas?

## Dados

Rode a célula abaixo: ela carrega os 120 blocos da base. Na parte 1 usamos só os 30 da planilha.

In [ ]:
import io

# Os 120 blocos da base sintética (não precisa ler linha a linha).
# Colunas: bloco, coordenadas (km), área (ha) e distância por estrada até as fábricas A e B (km).
base = pd.read_csv(io.StringIO("""bloco,x_km,y_km,area_ha,dist_A,dist_B
B001,21.3,88.7,362,45.2,171.4
B002,115.7,142.3,247,113.1,115.9
B003,117.8,140.2,347,113.1,112.4
B004,61.2,112.5,417,43.1,134.1
B005,96.4,58.4,370,60.7,69.7
B006,44.4,104.0,431,34.1,148.8
B007,71.0,114.4,209,49.3,124.6
B008,49.3,86.3,286,11.0,135.3
B009,138.8,92.1,565,110.1,44.1
B010,64.5,80.0,205,12.3,114.2
B011,102.8,82.5,339,62.2,68.0
B012,117.7,152.0,802,124.1,126.8
B013,156.9,73.1,551,132.8,19.3
B014,135.7,88.2,466,105.4,41.1
B015,93.9,45.9,558,67.3,75.2
B016,134.0,107.6,509,108.8,65.2
B017,149.4,55.3,564,126.8,6.2
B018,166.1,77.9,571,144.4,31.3
B019,154.6,75.9,279,129.6,21.5
B020,135.6,70.2,253,105.6,22.9
B021,50.5,98.9,360,25.2,138.9
B022,58.7,101.2,530,28.0,130.3
B023,154.1,99.0,477,131.2,51.0
B024,30.4,105.3,127,45.9,166.3
B025,28.7,78.5,213,34.3,159.5
B026,91.1,78.5,365,47.0,80.3
B027,49.7,86.4,465,10.8,134.8
B028,181.3,72.0,311,164.6,43.6
B029,65.9,92.7,482,21.8,117.3
B030,157.3,52.8,376,137.6,13.3
B031,48.1,89.3,258,15.1,137.8
B032,184.2,88.7,239,168.3,58.1
B033,164.4,80.5,440,142.3,32.6
B034,69.0,93.2,268,25.0,113.8
B035,103.7,168.2,749,131.0,153.0
B036,90.9,29.1,1002,80.9,86.7
B037,106.7,135.3,435,98.4,112.9
B038,69.7,70.4,460,22.8,105.3
B039,99.7,113.9,534,72.9,95.8
B040,97.1,56.0,152,63.0,68.9
B041,63.6,114.4,543,46.1,132.7
B042,119.6,146.7,94,120.8,119.5
B043,115.2,73.3,745,78.8,48.4
B044,94.8,30.7,839,82.3,81.3
B045,39.4,88.3,335,22.9,148.4
B046,77.1,74.3,110,29.6,96.6
B047,13.9,89.4,158,54.8,181.0
B048,152.6,56.4,550,130.6,5.8
B049,67.7,39.6,422,55.0,110.2
B050,50.7,95.6,266,21.0,137.1
B051,93.9,136.1,498,88.7,123.0
B052,112.8,109.9,202,84.6,81.0
B053,58.4,99.5,521,25.7,129.7
B054,153.7,63.4,368,130.1,6.5
B055,127.8,129.2,358,114.2,94.4
B056,160.2,45.3,1461,144.0,23.3
B057,34.7,90.0,267,29.4,154.9
B058,47.4,61.5,338,26.0,133.4
B059,109.6,129.9,667,96.1,105.0
B060,186.9,95.8,205,172.7,66.8
B061,165.8,97.3,305,145.7,52.7
B062,92.6,38.2,219,73.1,79.8
B063,24.7,105.2,200,51.3,173.2
B064,43.4,93.6,328,23.3,145.3
B065,164.9,60.8,223,145.1,19.4
B066,173.9,39.3,369,163.4,41.1
B067,36.3,95.3,559,31.4,154.8
B068,94.7,40.8,306,72.5,76.1
B069,11.8,81.0,304,56.2,181.7
B070,73.3,90.4,148,27.3,107.3
B071,94.8,31.8,290,81.3,80.6
B072,189.4,75.6,347,174.8,55.1
B073,195.3,66.4,441,183.3,59.5
B074,29.5,105.1,470,46.5,167.3
B075,39.6,77.2,715,20.4,145.3
B076,94.7,44.4,872,69.3,74.7
B077,114.6,147.3,887,116.9,122.5
B078,64.9,107.2,1713,37.6,126.5
B079,39.1,72.5,348,22.8,145.1
B080,131.4,133.3,199,121.1,98.3
B081,34.9,83.2,599,26.4,152.6
B082,94.6,59.5,493,58.0,72.0
B083,166.2,88.2,199,144.9,42.3
B084,146.2,153.8,297,152.6,122.0
B085,168.9,103.4,891,151.2,61.5
B086,113.9,43.6,283,89.9,51.6
B087,99.6,50.0,249,69.9,66.8
B088,96.4,120.2,239,75.1,104.8
B089,101.0,76.8,493,60.0,67.3
B090,89.0,31.6,787,76.9,87.5
B091,193.3,87.6,240,180.1,66.7
B092,153.2,56.4,334,131.3,6.3
B093,103.0,73.1,492,63.0,63.5
B094,153.5,90.8,751,128.8,40.3
B095,112.2,82.7,619,74.5,57.3
B096,14.5,73.6,129,53.2,177.0
B097,144.9,38.7,485,128.7,28.5
B098,124.6,131.2,190,112.4,98.3
B099,142.8,72.4,348,114.6,18.7
B100,128.8,126.5,765,113.4,90.7
B101,62.6,77.8,777,10.3,115.9
B102,36.2,110.1,295,46.2,161.6
B103,81.4,113.4,618,55.4,113.0
B104,83.1,41.5,198,62.0,90.2
B105,125.3,126.6,621,109.7,92.3
B106,146.4,68.6,223,119.8,12.1
B107,172.2,73.5,362,152.6,33.7
B108,88.9,77.1,193,44.2,82.5
B109,34.0,84.3,236,27.9,154.1
B110,61.9,80.0,374,9.0,117.4
B111,207.2,83.8,693,197.9,80.5
B112,73.8,86.5,434,25.8,104.9
B113,24.1,88.1,342,41.5,167.7
B114,79.4,100.0,599,41.0,105.4
B115,109.5,55.7,202,77.6,52.9
B116,146.8,113.5,251,127.1,69.7
B117,21.6,94.3,263,47.3,172.8
B118,98.6,125.5,248,81.9,108.3
B119,127.0,104.8,528,99.1,65.5
B120,143.8,101.6,338,118.8,54.7
"""))

# Posição (x, y em km) das duas fábricas atuais e dos três locais candidatos a uma nova fábrica
posicao = {
    "A": (55, 80),
    "B": (150, 60),
    "Local 1": (110, 125),
    "Local 2": (30, 40),
    "Local 3": (175, 125),
}

# Distância por estrada = linha reta × 1.3 (sinuosidade); a mesma regra das colunas dist_A e dist_B
SINUOSIDADE = 1.3

# Os 30 blocos da planilha 03_blocos_fabricas.xlsx
blocos_da_planilha = ["B006", "B007", "B010", "B011", "B019", "B028", "B031", "B033", "B038", "B039", "B041", "B042", "B045", "B046", "B053", "B070", "B071", "B076", "B078", "B084", "B087", "B094", "B095", "B098", "B100", "B102", "B104", "B105", "B110", "B115"]

print(len(base), "blocos,", br(base.area_ha.sum(), 0), "ha")

Para o modelo, os dados ficam em **listas** e **dicionários**:
um nome para o conjunto e um valor para cada item.

In [ ]:
def preparar(tabela):
    """Lista de blocos, área de cada bloco e distância de cada par (bloco, fábrica)."""
    blocos = list(tabela.bloco)
    area = dict(zip(tabela.bloco, tabela.area_ha))
    distancia = {}
    for linha in tabela.itertuples():
        distancia[linha.bloco, "A"] = linha.dist_A
        distancia[linha.bloco, "B"] = linha.dist_B
    return blocos, area, distancia


# Parte 1: só os 30 blocos da planilha
dados_30 = base[base.bloco.isin(blocos_da_planilha)]
blocos, area, distancia = preparar(dados_30)
fabricas = ["A", "B"]
cota = {"A": 0.40, "B": 0.55}
area_total = sum(area.values())
print(len(blocos), "blocos ·", br(area_total, 0), "ha")

## Validações macro

As mesmas contas das abas `1_estimar` e `4_comparar` da planilha, escritas uma vez e usadas em todas as partes.
**Você não precisa entender este código:** basta rodar.

In [ ]:
def conferir_antes(blocos, area, distancia, fabricas, cota):
    """Contas macro antes de resolver: o problema tem solução? Entre que valores a DMT vai ficar?"""
    area_total = sum(area.values())
    mais_proxima = {b: min(fabricas, key=lambda f: distancia[b, f]) for b in blocos}
    piso = sum(area[b] * distancia[b, mais_proxima[b]] for b in blocos) / area_total
    teto = sum(area[b] * max(distancia[b, f] for f in fabricas) for b in blocos) / area_total
    print("Área total:", br(area_total, 0), "ha")
    print("Soma das cotas:", br(100 * sum(cota.values()), 0), "%",
          "→ cabe em 100%" if sum(cota.values()) <= 1 else "→ PASSA DE 100%: o problema não tem solução")
    for f in fabricas:
        pela_proxima = sum(area[b] for b in blocos if mais_proxima[b] == f)
        minimo = cota[f] * area_total
        falta = max(0, minimo - pela_proxima)
        print(f"Fábrica {f}: pela mais próxima recebe {br(pela_proxima, 0)} ha; precisa de {br(minimo, 0)} ha;",
              f"faltam {br(falta, 0)} ha" if falta > 0 else "cota já atendida")
    print("Piso (todos na mais próxima):", br(piso, 1), "km · Teto (todos na mais longe):", br(teto, 1), "km")
    return piso, teto, mais_proxima


def conferir_depois(modelo, blocos, area, fabricas, cota, piso, teto):
    """Contas macro depois de resolver: a solução respeita tudo o que deveria?"""
    area_total = sum(area.values())
    x = {(b, f): pyo.value(modelo.x[b, f]) for b in blocos for f in fabricas}
    dmt = pyo.value(modelo.dmt)
    area_recebida = {f: sum(area[b] * round(x[b, f]) for b in blocos) for f in fabricas}
    linhas = [
        ("Blocos em exatamente uma fábrica", br(sum(abs(sum(x[b, f] for f in fabricas) - 1) < 1e-6 for b in blocos), 0), f"= {len(blocos)}",
         sum(abs(sum(x[b, f] for f in fabricas) - 1) < 1e-6 for b in blocos) == len(blocos)),
        ("Variáveis fora de 0 ou 1", br(sum(1e-6 < v < 1 - 1e-6 for v in x.values()), 0), "= 0",
         all(not (1e-6 < v < 1 - 1e-6) for v in x.values())),
        ("Área designada (ha)", br(sum(area_recebida.values()), 0), f"= {br(area_total, 0)}",
         abs(sum(area_recebida.values()) - area_total) < 0.5),
    ]
    for f in fabricas:
        folga = area_recebida[f] - cota[f] * area_total
        linhas.append((f"Folga da cota da {f} (ha)", br(folga, 0), "≥ 0", folga >= -1e-6))
    linhas += [
        ("DMT ótima (km)", br(dmt, 2), f"≥ piso ({br(piso, 1)})", dmt >= piso - 1e-6),
        ("DMT ótima (km)", br(dmt, 2), f"≤ teto ({br(teto, 1)})", dmt <= teto + 1e-6),
    ]
    return pd.DataFrame(
        [(nome, valor, regra, "OK" if ok else "VERIFICAR") for nome, valor, regra, ok in linhas],
        columns=["validação", "valor", "regra", "ok?"],
    )

# Parte 1: os 30 blocos da planilha

## 1. Estimar antes de modelar

In [ ]:
piso, teto, mais_proxima = conferir_antes(blocos, area, distancia, fabricas, cota)

## 2. Modelar

$$
\begin{aligned}
\min \quad & \text{DMT} = \frac{1}{A}\sum_{b}\sum_{f} a_b\, d_{bf}\, x_{bf} && \text{(km)}\\
\text{sujeito a} \quad & \sum_{f} x_{bf} = 1 && \text{para cada bloco } b\\
& \sum_{b} a_b\, x_{bf} \ge \text{cota}_f \cdot A && \text{para cada fábrica } f\\
& x_{bf} \in \{0, 1\}
\end{aligned}
$$

$x_{bf}$ = 1 se o bloco $b$ vai para a fábrica $f$ (0 se não); $a_b$ = área do bloco (ha);
$d_{bf}$ = distância (km); $A$ = área total (ha). Cada linha com $\sum$ ou
"para cada" vira **uma** célula de código, sejam 30 ou 120 blocos.

**Variáveis de decisão:** $x_{bf} \in \{0, 1\}$, uma para cada par (bloco, fábrica). A novidade é o `domain=pyo.Binary`: o *solver* só aceita 0 ou 1.

In [ ]:
modelo = pyo.ConcreteModel("blocos_fabricas")

# Uma variável sim/não para cada par (bloco, fábrica)
modelo.x = pyo.Var(blocos, fabricas, domain=pyo.Binary)
print("Variáveis:", len(modelo.x))

**Objetivo:** $\min\ \text{DMT} = \frac{1}{A}\sum_b \sum_f a_b\, d_{bf}\, x_{bf}$

In [ ]:
@modelo.Objective(sense=pyo.minimize)
def dmt(m):
    return sum(area[b] * distancia[b, f] * m.x[b, f] for b in blocos for f in fabricas) / area_total

**Cada bloco em exatamente uma fábrica**, para cada bloco $b$: $\sum_f x_{bf} = 1$. Uma célula gera as 30 restrições. (exemplo pronto)

In [ ]:
@modelo.Constraint(blocos)
def uma_fabrica(m, b):
    return sum(m.x[b, f] for f in fabricas) == 1

**Cota mínima**, para cada fábrica $f$: $\sum_b a_b\, x_{bf} \ge \text{cota}_f \cdot A$. Dica: a soma percorre os blocos, e o lado direito usa o dicionário `cota`.

In [ ]:
@modelo.Constraint(fabricas)
def cota_minima(m, f):
    return sum(area[b] * m.x[b, f] for b in ____) >= ____ * area_total

## 3. Resolver

In [ ]:
otimo = resolver(modelo)

Para qual fábrica foi cada bloco? A coluna `mudou?` marca os blocos que **não** foram para a mais próxima.

In [ ]:
def designacao(modelo, blocos, area, distancia, fabricas, mais_proxima):
    """Tabela com a fábrica escolhida para cada bloco."""
    linhas = []
    for b in blocos:
        f = max(fabricas, key=lambda f: pyo.value(modelo.x[b, f]))
        linhas.append({"bloco": b, "área (ha)": area[b], "fábrica": f, "distância (km)": distancia[b, f],
                       "mais próxima": mais_proxima[b], "mudou?": "sim" if f != mais_proxima[b] else ""})
    return pd.DataFrame(linhas)


tabela_30 = designacao(modelo, blocos, area, distancia, fabricas, mais_proxima)
tabela_30[tabela_30["mudou?"] == "sim"]

## 4. Comparar

As validações da aba `4_comparar`. A DMT tem de ser a mesma da planilha.

In [ ]:
conferir_depois(modelo, blocos, area, fabricas, cota, piso, teto)

In [ ]:
print("Piso:", br(piso, 2), "km · DMT ótima:", br(otimo, 2), "km")
print("Preço da cota:", br(otimo - piso, 2), "km (", br(100 * (otimo - piso) / piso, 1), "% )")

# Parte 2: os 120 blocos

Com os 120 blocos são **240 variáveis**, acima do limite de 200 do Solver do Excel. Aqui o modelo é o mesmo:
juntamos as células da parte 1 numa função, para usar com qualquer lista de blocos e qualquer cota.

In [ ]:
def montar_modelo(blocos, area, distancia, fabricas, cota):
    """O mesmo modelo da parte 1, para qualquer conjunto de blocos, fábricas e cotas."""
    area_total = sum(area.values())
    modelo = pyo.ConcreteModel("blocos_fabricas")
    modelo.x = pyo.Var(blocos, fabricas, domain=pyo.Binary)

    @modelo.Objective(sense=pyo.minimize)
    def dmt(m):
        return sum(area[b] * distancia[b, f] * m.x[b, f] for b in blocos for f in fabricas) / area_total

    @modelo.Constraint(blocos)
    def uma_fabrica(m, b):
        return sum(m.x[b, f] for f in fabricas) == 1

    # Copie aqui a cota mínima que você completou na parte 1
    @modelo.Constraint(fabricas)
    def cota_minima(m, f):
        return sum(area[b] * m.x[b, f] for b in ____) >= ____ * area_total

    return modelo

In [ ]:
blocos_120, area_120, distancia_120 = preparar(base)
piso_120, teto_120, proxima_120 = conferir_antes(blocos_120, area_120, distancia_120, fabricas, cota)

In [ ]:
modelo_120 = montar_modelo(blocos_120, area_120, distancia_120, fabricas, cota)
print("Variáveis:", len(modelo_120.x), "· Restrições:", len(list(modelo_120.component_data_objects(pyo.Constraint))))
otimo_120 = resolver(modelo_120)
conferir_depois(modelo_120, blocos_120, area_120, fabricas, cota, piso_120, teto_120)

### O mapa

Cada ponto é um bloco (tamanho proporcional à área), na cor da fábrica escolhida. Os blocos com contorno
preto **mudaram** de fábrica por causa da cota. Pode só rodar.

In [ ]:
import matplotlib.pyplot as plt

CORES = {"A": "#2a78d6", "B": "#eb6834", "Local 1": "#1baf7a", "Local 2": "#1baf7a", "Local 3": "#1baf7a"}


def mapa(modelo, tabela, fabricas, mais_proxima, titulo):
    """Blocos na cor da fábrica escolhida; contorno preto = não foi para a mais próxima."""
    fig, ax = plt.subplots(figsize=(8, 6))
    for linha in tabela.itertuples():
        f = max(fabricas, key=lambda f: pyo.value(modelo.x[linha.bloco, f]))
        mudou = f != mais_proxima[linha.bloco]
        ax.scatter(linha.x_km, linha.y_km, s=linha.area_ha / 6, color=CORES[f], alpha=0.75,
                   edgecolor="black" if mudou else "white", linewidth=1.8 if mudou else 0.5)
    for f in fabricas:
        x, y = posicao[f]
        ax.scatter(x, y, marker="s", s=180, color=CORES[f], edgecolor="black", zorder=3)
        ax.annotate(f"Fábrica {f}" if len(f) == 1 else f, (x, y), xytext=(8, 8), textcoords="offset points", fontweight="bold")
    ax.set_title(titulo)
    ax.set_xlabel("x (km)")
    ax.set_ylabel("y (km)")
    ax.set_aspect("equal")
    plt.show()


mapa(modelo_120, base, fabricas, proxima_120, f"120 blocos com cota: DMT de {br(otimo_120, 1)} km")

# Parte 3: cenários

Agora é só trocar os dados e chamar a mesma função. Dois tipos de pergunta:

- **E se a cota mudasse?** Sem cota, cada bloco vai para a mais próxima.
- **Onde construir uma terceira fábrica?** Cada local candidato entra como mais uma fábrica, e as três
  precisam de pelo menos 30% da área.

In [ ]:
def distancia_ate(tabela, local):
    """Distância por estrada (km) de cada bloco até um local: linha reta × sinuosidade."""
    x, y = posicao[local]
    return {linha.bloco: round(((linha.x_km - x) ** 2 + (linha.y_km - y) ** 2) ** 0.5 * SINUOSIDADE, 1)
            for linha in tabela.itertuples()}


def rodar_cenario(nome, fabricas_cenario, cota_cenario):
    """Monta, resolve e resume um cenário com os 120 blocos."""
    dist = dict(distancia_120)
    for f in fabricas_cenario:
        if f not in ("A", "B"):
            for b, d in distancia_ate(base, f).items():
                dist[b, f] = d
    m = montar_modelo(blocos_120, area_120, dist, fabricas_cenario, cota_cenario)
    pyo.SolverFactory("appsi_highs").solve(m)
    areas = {f: sum(area_120[b] * round(pyo.value(m.x[b, f])) for b in blocos_120) for f in fabricas_cenario}
    return {"cenário": nome, "DMT (km)": round(pyo.value(m.dmt), 1),
            **{f"área {f} (ha)": areas[f] for f in fabricas_cenario}}


cenarios = [
    rodar_cenario("Sem cota", ["A", "B"], {"A": 0, "B": 0}),
    rodar_cenario("Atual: A ≥ 40%, B ≥ 55%", ["A", "B"], cota),
    rodar_cenario("Nova fábrica no Local 1", ["A", "B", "Local 1"], {"A": 0.3, "B": 0.3, "Local 1": 0.3}),
    rodar_cenario("Nova fábrica no Local 2", ["A", "B", "Local 2"], {"A": 0.3, "B": 0.3, "Local 2": 0.3}),
    rodar_cenario("Nova fábrica no Local 3", ["A", "B", "Local 3"], {"A": 0.3, "B": 0.3, "Local 3": 0.3}),
]
tabela = pd.DataFrame(cenarios)
colunas_area = [c for c in tabela.columns if c.startswith("área")]
tabela[colunas_area] = tabela[colunas_area].map(lambda v: "" if pd.isna(v) else br(v, 0))
tabela

## 5. Perguntas

Dê dois cliques nesta célula para escrever as suas respostas.

**1. A DMT da parte 1 bateu com a planilha? Quais blocos mudaram de fábrica, e o que eles têm em comum?**

> Sua resposta:

**2. Com os 120 blocos, quanto a cota custou em DMT? Quantas variáveis e restrições o modelo teve?**

> Sua resposta:

**3. Qual o melhor local para a terceira fábrica? Antes de rodar, dava para chutar olhando o mapa?**

> Sua resposta: